# 🎤 STT avec Faster-Whisper

Ce notebook permet d'expérimenter avec **faster-whisper** pour la transcription audio.

## Installation

In [ ]:
# Installation des dépendances
!pip install faster-whisper

## Configuration du modèle

Modèles disponibles (du plus léger au plus précis) :
- `tiny` (~75 Mo) - très rapide, moins précis
- `base` (~145 Mo) - bon compromis pour un PoC
- `small` (~488 Mo) - meilleure qualité
- `medium` (~1.5 Go) - très bonne qualité
- `large-v3` (~3 Go) - meilleure qualité possible

In [ ]:
from faster_whisper import WhisperModel

# Configuration
MODEL_SIZE = "base"  # Changer selon tes besoins
DEVICE = "cpu"       # "cuda" si tu as un GPU NVIDIA
COMPUTE_TYPE = "int8"  # "float16" pour GPU, "int8" pour CPU

print(f"Chargement du modèle {MODEL_SIZE}...")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print("Modèle chargé !")

## Test avec un fichier audio

Place un fichier audio (WAV, MP3, etc.) dans le dossier `ia/` ou indique le chemin complet.

In [ ]:
import time

# Chemin vers ton fichier audio de test
AUDIO_FILE = "test.wav"  # À modifier selon ton fichier

# Transcription
print(f"Transcription de {AUDIO_FILE}...")
start_time = time.time()

segments, info = model.transcribe(
    AUDIO_FILE,
    language="fr",  # Force le français
    beam_size=5
)

# Récupération du texte complet
transcription = " ".join([segment.text for segment in segments])

elapsed_time = time.time() - start_time

print(f"\n📊 Résultats :")
print(f"  - Langue détectée : {info.language}")
print(f"  - Probabilité langue : {info.language_probability:.2%}")
print(f"  - Durée audio : {info.duration:.2f}s")
print(f"  - Temps de traitement : {elapsed_time:.2f}s")
print(f"  - Ratio (temps réel) : {elapsed_time / info.duration:.2f}x")
print(f"\n📝 Transcription :\n{transcription}")

## Test avec enregistrement micro (optionnel)

Si tu veux tester directement avec ton micro.

In [ ]:
# Installation pour l'enregistrement audio
!pip install sounddevice soundfile

In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np

# Configuration de l'enregistrement
SAMPLE_RATE = 16000  # 16kHz comme recommandé
DURATION = 5  # Durée en secondes

print(f"🎙️ Enregistrement pendant {DURATION} secondes...")
print("Parle maintenant !")

# Enregistrement
audio = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='int16')
sd.wait()  # Attend la fin de l'enregistrement

# Sauvegarde temporaire
temp_file = "temp_recording.wav"
sf.write(temp_file, audio, SAMPLE_RATE)
print(f"✅ Enregistrement sauvegardé dans {temp_file}")

In [ ]:
# Transcription de l'enregistrement
import time

print("Transcription en cours...")
start_time = time.time()

segments, info = model.transcribe("temp_recording.wav", language="fr")
transcription = " ".join([segment.text for segment in segments])

elapsed_time = time.time() - start_time

print(f"\n⏱️ Temps de traitement : {elapsed_time:.2f}s")
print(f"📝 Transcription : {transcription}")

## Fonction utilitaire pour l'API

Voici une fonction prête à être utilisée dans l'API FastAPI.

In [ ]:
from faster_whisper import WhisperModel
from typing import Tuple
import io

class STTService:
    """Service de Speech-to-Text avec Faster-Whisper"""
    
    def __init__(self, model_size: str = "base", device: str = "cpu"):
        compute_type = "float16" if device == "cuda" else "int8"
        self.model = WhisperModel(model_size, device=device, compute_type=compute_type)
    
    def transcribe(self, audio_path: str, language: str = "fr") -> Tuple[str, dict]:
        """
        Transcrit un fichier audio en texte.
        
        Args:
            audio_path: Chemin vers le fichier audio
            language: Code langue (fr, en, etc.)
            
        Returns:
            Tuple (texte, métadonnées)
        """
        segments, info = self.model.transcribe(audio_path, language=language)
        text = " ".join([s.text for s in segments]).strip()
        
        metadata = {
            "language": info.language,
            "language_probability": info.language_probability,
            "duration": info.duration
        }
        
        return text, metadata

# Test
# stt = STTService(model_size="base")
# text, meta = stt.transcribe("test.wav")
# print(text, meta)

## Notes

- Le premier chargement du modèle télécharge les poids (~145 Mo pour `base`)
- Les poids sont cachés dans `~/.cache/huggingface/`
- Sur CPU, le modèle `base` traite ~0.5x temps réel (5s audio = ~10s traitement)
- Sur GPU CUDA, c'est ~10x plus rapide